# 04 — Gate 2 with RELEASED detectors: is the indirect collapse real?
Gate 2 v2 showed e5+XGBoost (trained only on deepset) is a *domain* detector: it flags every BIPIA document (benign FPR 100%, indirect AUROC 0.62). That's an artefact, not a measurement.

Here we swap in detectors that have actually seen indirect/document attacks and were NOT trained on deepset, so they can't have its domain bias:
- `protectai/deberta-v3-base-prompt-injection-v2` (open)
- `meta-llama/Llama-Prompt-Guard-2-86M` (gated -- accept Meta's licence on the model page + provide an HF token below)

Same frozen-threshold direct->indirect transfer. **The score diagnostic is baked in** so we immediately see whether each detector discriminates on indirect docs.

## Session bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first, every session) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Dependencies

In [2]:
!pip install -q transformers sentencepiece datasets
import torch; print('CUDA:', torch.cuda.is_available())

CUDA: True


## HF token (needed only for the gated Prompt-Guard-2)
1) On huggingface.co, open `meta-llama/Llama-Prompt-Guard-2-86M` and accept the licence. 2) Paste a token below (blank = skip Prompt-Guard-2, still run ProtectAI).

In [3]:
from huggingface_hub import login
import getpass
_t = getpass.getpass('HF token (blank to skip gated model): ').strip()
if _t: login(token=_t); print('logged in')
else: print('no token - will skip Prompt-Guard-2')

HF token (blank to skip gated model): ··········
logged in


## Load source (deepset direct) + realistic indirect (BIPIA)

In [4]:
import importlib, data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import load_deepset, load_bipia_indirect
from detectors import score_released, threshold_at_fpr, auroc
from metrics import all_metrics, severity_S, bootstrap_ci
import numpy as np, pandas as pd, os

deepset = load_deepset()
bipia   = load_bipia_indirect(n_attack=400)
print('deepset', len(deepset), '| bipia indirect', len(bipia))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/116 [00:00<?, ? examples/s]

Cloning microsoft/BIPIA ...
BIPIA indirect: 1097 contexts, 150 text attacks
deepset 662 | bipia indirect 1177


## Score with each released detector (cached to data/)
Wrapped in try/except: if Prompt-Guard-2 is gated/unavailable, ProtectAI still runs.

In [ ]:
DETECTORS = {
  'protectai_v2':   'protectai/deberta-v3-base-prompt-injection-v2',
  'prompt_guard_2': 'meta-llama/Llama-Prompt-Guard-2-86M',
}
os.makedirs('data', exist_ok=True)
scores = {}
for tag, name in DETECTORS.items():
    fp_d, fp_b = f'data/score_{tag}_deepset.npy', f'data/score_{tag}_bipia.npy'
    try:
        if os.path.exists(fp_d) and os.path.exists(fp_b):
            print('load cached', tag); pd_, pb_ = np.load(fp_d), np.load(fp_b)
        else:
            print('scoring', tag, name)
            pd_ = score_released(deepset.text.tolist(), name)
            pb_ = score_released(bipia.text.tolist(),   name)
            np.save(fp_d, pd_); np.save(fp_b, pb_)
        scores[tag] = (pd_, pb_)
        print(f'  {tag}: ok')
    except Exception as e:
        print(f'  [skip] {tag}: {e}')
print('\nscored detectors:', list(scores.keys()))

scoring protectai_v2 protectai/deberta-v3-base-prompt-injection-v2


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  protectai/deberta-v3-base-prompt-injection-v2: id2label={0: 'SAFE', 1: 'INJECTION'} -> attack class index 1


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  protectai/deberta-v3-base-prompt-injection-v2: id2label={0: 'SAFE', 1: 'INJECTION'} -> attack class index 1


## Metrics + score diagnostic per detector
Frozen threshold at 1% FPR on deepset benigns. The diagnostic (indirect benign FPR, indirect-only AUROC) tells us if the detector actually discriminates.

In [ ]:
yd = deepset.label.values; yb = bipia.label.values
summary = []
for tag, (pd_, pb_) in scores.items():
    t = threshold_at_fpr(pd_[yd==0], target_fpr=0.01)
    m_s = all_metrics(yd, pd_, t=t); m_s['AUROC'] = auroc(yd, pd_)
    m_t = all_metrics(yb, pb_, t=t); m_t['AUROC'] = auroc(yb, pb_)
    ind_ben_fpr = (pb_[yb==0] >= t).mean()
    ind_atk_tpr = (pb_[yb==1] >= t).mean()
    ind_auroc   = auroc(yb, pb_)
    summary.append({
        'detector': tag, 't': round(t,4),
        'src_AUROC': round(m_s['AUROC'],3), 'tgt_AUROC': round(m_t['AUROC'],3),
        'ind_benFPR': round(float(ind_ben_fpr),3), 'ind_atkTPR': round(float(ind_atk_tpr),3),
        'src_FNR': round(m_s['FNR'],3), 'tgt_FNR': round(m_t['FNR'],3),
        'tgt_S': (round(m_t['S'],3) if not np.isnan(m_t['S']) else 'NaN'),
        'tgt_FNCR': round(m_t['FNCR'],3), 'tgt_ECEatk': round(m_t['ECE_atk'],3),
        'tgt_misses': m_t['n_misses'],
    })
summ = pd.DataFrame(summary)
print(summ.to_string(index=False))

## Score distributions (the tell)
If indirect benign and attack scores overlap and pile up high -> domain flooding (like e5+XGB). If benigns sit low and a few attacks slip high -> real S signal.

In [ ]:
import matplotlib.pyplot as plt
n = len(scores)
if n:
    fig, axes = plt.subplots(1, n, figsize=(6*n,4), squeeze=False)
    for ax,(tag,(pd_,pb_)) in zip(axes[0], scores.items()):
        ax.hist(pb_[yb==0], bins=30, alpha=.6, density=True, label='indirect benign')
        ax.hist(pb_[yb==1], bins=30, alpha=.6, density=True, label='indirect attack')
        ax.set_title(tag); ax.set_xlabel('p=P(attack)'); ax.legend()
    plt.tight_layout(); os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/gate2_released_scores.png', dpi=150); plt.show()
summ.to_csv('reports/gate2_released_summary.csv', index=False)
print('saved figures/gate2_released_scores.png + reports/gate2_released_summary.csv')

## Verdict

In [ ]:
vlines = ['READING THE TABLE:']
for r in summary:
    tag = r['detector']; vlines.append(f'[{tag}]')
    if r['ind_benFPR'] > 0.5:
        vlines.append(f"  indirect benign FPR {r['ind_benFPR']:.0%} -> still FLOODING; not a fair instrument here.")
    elif r['tgt_AUROC'] < 0.7:
        vlines.append(f"  weak indirect discrimination (tgt AUROC {r['tgt_AUROC']}); inconclusive.")
    else:
        vlines.append(f"  discriminates on indirect (benFPR {r['ind_benFPR']:.0%}, AUROC {r['tgt_AUROC']}).")
        if r['tgt_S'] != 'NaN' and float(r['tgt_S']) > 0.5 and r['tgt_misses'] >= 10:
            vlines.append(f"  AND missed indirect attacks are confident: S={r['tgt_S']} over {r['tgt_misses']} misses.")
            vlines.append('  -> REAL high-confidence-miss signal on a purpose-built detector. Headline candidate.')
        else:
            vlines.append(f"  few/low-confidence misses (S={r['tgt_S']}, misses={r['tgt_misses']}) -> transports cleanly here;")
            vlines.append('     calibration story likely lives in harder cells (jailbreak / over-defense).')
verdict_text = '\n'.join(vlines)
print(verdict_text)

## Persist results to reports/RESULTS_LOG.md

In [ ]:
from reslog import log_result
body = 'Frozen-threshold direct->indirect, released detectors.\n\n' + summ.to_string(index=False) + '\n\n' + verdict_text
log_result('Gate 2 released detectors (Prompt-Guard-2 / ProtectAI-v2)', body,
           csv_df=summ, csv_name='gate2_released_summary.csv')

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Gate 2 released detectors (Prompt-Guard-2, ProtectAI-v2): direct->indirect transfer + score diagnostic"
!git push
print('Pushed.')